# Watermarking DCT Domain

Menyisipkan watermark ke dalam gambar menggunakan transformasi **Discrete Cosine Transform (DCT)**.

**NIM:** 18224014

## Cara Kerja
1. Gambar dibagi menjadi blok-blok 8x8 pixel
2. Setiap blok ditransformasi ke domain frekuensi menggunakan DCT 2D
3. Bit watermark disisipkan pada koefisien mid-frequency [4][4]
4. Blok dikembalikan ke domain spasial menggunakan Inverse DCT
5. Watermark dapat diekstrak kembali dari gambar yang sudah di-watermark

## Setup Library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from scipy.fftpack import dct, idct
import io

print('Libraries loaded!')

## Step 1: Load Foto dan Lihat Aslinya

In [ ]:
IMAGE_PATH = 'original.png'
WATERMARK_TEXT = '18224014'
ALPHA = 25          # kekuatan watermark
BLOCK_SIZE = 8      # ukuran blok DCT

img = Image.open(IMAGE_PATH).convert('RGB')
img_array = np.array(img, dtype=np.float64)

print(f'Ukuran gambar: {img_array.shape}')
print(f'Min pixel: {img_array.min():.0f}, Max pixel: {img_array.max():.0f}')

plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.title('Gambar Original')
plt.axis('off')
plt.tight_layout()
plt.show()

## Step 2: Konversi Watermark ke Bit

In [ ]:
def text_to_bits(text):
    """Mengubah teks menjadi deretan bit (0 dan 1)"""
    bits = []
    for char in text:
        b = format(ord(char), '08b')
        bits.extend([int(x) for x in b])
    return bits

def bits_to_text(bits):
    """Mengubah deretan bit kembali menjadi teks"""
    chars = []
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        if len(byte) < 8:
            break
        char = chr(int(''.join(map(str, byte)), 2))
        chars.append(char)
    return ''.join(chars)

watermark_bits = text_to_bits(WATERMARK_TEXT)
length_bits = [int(x) for x in format(len(watermark_bits), '016b')]
all_bits = length_bits + watermark_bits

print(f'Watermark text   : "{WATERMARK_TEXT}"')
print(f'Jumlah karakter  : {len(WATERMARK_TEXT)}')
print(f'Jumlah bit data  : {len(watermark_bits)}')
print(f'Header (16 bit)  : {length_bits}')
print(f'Total bit embed  : {len(all_bits)}')
print(f'Contoh bit "1"   : {text_to_bits("1")}')

## Step 3: Visualisasi DCT pada Satu Blok 8x8

In [ ]:
# Ambil satu blok 8x8 dari pojok kiri atas gambar
sample_block = img_array[:8, :8, 0]  # channel merah
dct_block = dct(dct(sample_block.T, norm='ortho').T, norm='ortho')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im0 = axes[0].imshow(sample_block, cmap='gray')
axes[0].set_title('Blok 8x8 Original (Pixel Domain)')
plt.colorbar(im0, ax=axes[0])
for i in range(8):
    for j in range(8):
        axes[0].text(j, i, f'{sample_block[i,j]:.0f}', ha='center', va='center', fontsize=7, color='red')

im1 = axes[1].imshow(np.log(np.abs(dct_block) + 1), cmap='hot')
axes[1].set_title('Blok 8x8 setelah DCT (Frequency Domain)')
plt.colorbar(im1, ax=axes[1])
axes[1].add_patch(patches.Rectangle((3.5, 3.5), 1, 1, linewidth=2, edgecolor='cyan', facecolor='none'))
axes[1].text(4, 4, 'Embed\ndi sini', ha='center', va='center', fontsize=7, color='cyan')

plt.suptitle('Transformasi DCT: Pixel → Frekuensi', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Koefisien DCT[4][4] = {dct_block[4][4]:.4f}  ← titik penyisipan watermark')

## Step 4: Embed Watermark

In [ ]:
def embed_watermark(img_array, all_bits, alpha=25, block_size=8):
    """
    Menyisipkan watermark ke dalam gambar menggunakan DCT.
    
    Cara kerja:
    - Bit = 1 → koefisien DCT[4][4] dibuat positif (+ alpha)
    - Bit = 0 → koefisien DCT[4][4] dibuat negatif (- alpha)
    """
    channel = img_array[:, :, 0].copy()  # channel merah
    h, w = channel.shape
    bit_idx = 0
    watermarked = channel.copy()
    
    for i in range(0, h - block_size + 1, block_size):
        for j in range(0, w - block_size + 1, block_size):
            if bit_idx >= len(all_bits):
                break
            block = channel[i:i+block_size, j:j+block_size]
            dct_block = dct(dct(block.T, norm='ortho').T, norm='ortho')
            
            # Sisipkan bit ke koefisien mid-frequency
            if all_bits[bit_idx] == 1:
                dct_block[4][4] = abs(dct_block[4][4]) + alpha
            else:
                dct_block[4][4] = -(abs(dct_block[4][4]) + alpha)
            
            idct_block = idct(idct(dct_block.T, norm='ortho').T, norm='ortho')
            watermarked[i:i+block_size, j:j+block_size] = idct_block
            bit_idx += 1
        if bit_idx >= len(all_bits):
            break
    
    result = img_array.copy()
    result[:, :, 0] = np.clip(watermarked, 0, 255)
    return np.clip(result, 0, 255).astype(np.uint8)

watermarked_array = embed_watermark(img_array, all_bits, alpha=ALPHA, block_size=BLOCK_SIZE)
watermarked_img = Image.fromarray(watermarked_array)
watermarked_img.save('watermarked.png')
print(f'Watermark berhasil disisipkan!')
print(f'Gambar disimpan sebagai watermarked.png')

## Step 5: Bandingkan Original vs Watermarked

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img)
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(watermarked_img)
axes[1].set_title('Watermarked')
axes[1].axis('off')

# Difference (diperbesar 10x biar kelihatan)
diff = np.abs(img_array - watermarked_array.astype(np.float64))
diff_amplified = np.clip(diff * 10, 0, 255).astype(np.uint8)
axes[2].imshow(diff_amplified)
axes[2].set_title('Perbedaan (x10)')
axes[2].axis('off')

plt.suptitle('Perbandingan: Original vs Watermarked', fontsize=13)
plt.tight_layout()
plt.show()

# Hitung PSNR
mse = np.mean((img_array - watermarked_array.astype(np.float64)) ** 2)
psnr = 10 * np.log10(255**2 / mse) if mse > 0 else float('inf')
print(f'MSE  : {mse:.4f}')
print(f'PSNR : {psnr:.2f} dB  (semakin tinggi = semakin mirip aslinya, >30dB = bagus)')

## Step 6: Ekstrak Watermark

In [ ]:
def extract_watermark(img_array, watermark_length_chars, block_size=8):
    """
    Mengekstrak watermark dari gambar yang sudah di-watermark.
    
    Cara kerja:
    - DCT[4][4] positif → bit = 1
    - DCT[4][4] negatif → bit = 0
    """
    channel = img_array[:, :, 0].astype(np.float64)
    h, w = channel.shape
    extracted_bits = []
    total_bits_needed = 16 + watermark_length_chars * 8
    
    for i in range(0, h - block_size + 1, block_size):
        for j in range(0, w - block_size + 1, block_size):
            if len(extracted_bits) >= total_bits_needed:
                break
            block = channel[i:i+block_size, j:j+block_size]
            dct_block = dct(dct(block.T, norm='ortho').T, norm='ortho')
            extracted_bits.append(1 if dct_block[4][4] >= 0 else 0)
        if len(extracted_bits) >= total_bits_needed:
            break
    
    data_bits = extracted_bits[16:]  # skip header 16 bit
    return bits_to_text(data_bits)

# Load watermarked image dan ekstrak
wm_img = Image.open('watermarked.png').convert('RGB')
wm_array = np.array(wm_img)

extracted = extract_watermark(wm_array, len(WATERMARK_TEXT))

print(f'Watermark asli     : "{WATERMARK_TEXT}"')
print(f'Watermark diekstrak: "{extracted}"')
print(f'Berhasil           : {extracted == WATERMARK_TEXT}')

## Kesimpulan

- Watermark NIM **18224014** berhasil disisipkan ke gambar menggunakan DCT
- Gambar hasil watermarking secara visual **tidak dapat dibedakan** dari aslinya
- PSNR > 30 dB menunjukkan kualitas gambar tetap baik
- Watermark berhasil diekstrak kembali dengan tepat

### Keunggulan DCT vs LSB
| | DCT | LSB |
|---|---|---|
| Domain | Frekuensi | Spasial |
| Ketahanan JPEG | Lebih tahan | Mudah rusak |
| Kompleksitas | Lebih tinggi | Lebih simpel |
| Kualitas visual | Sangat baik | Baik |